In [1]:
   import os
   from dotenv import load_dotenv
   from sqlalchemy import create_engine, text

   load_dotenv()
   url = (f"postgresql+psycopg://{os.getenv('PG_USER')}:{os.getenv('PG_PASSWORD')}"
          f"@{os.getenv('PG_HOST')}:{os.getenv('PG_PORT')}/{os.getenv('PG_DB')}")
   engine = create_engine(url)

   with engine.connect() as con:
       print(con.execute(text("SELECT version()")).scalar())

PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


In [4]:
import os, pathlib
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, event, text

load_dotenv("../.env")
engine_pg = create_engine(
    f"postgresql+psycopg://{os.getenv('PG_USER')}:{os.getenv('PG_PASSWORD')}"
    f"@{os.getenv('PG_HOST')}:{os.getenv('PG_PORT')}/{os.getenv('PG_DB')}"
)
engine_sqlite = create_engine("sqlite:///../data/processed/schnee.db")

@event.listens_for(engine_sqlite, "connect")
def fk_an(dbapi_con, _):
    dbapi_con.execute("PRAGMA foreign_keys = ON")   # SQLite prüft Fremdschlüssel nur auf Wunsch

schema = pathlib.Path("../sql/schema.sql").read_text()

P = "../data/processed/"
gem = pd.read_csv(P + "gemeinden.csv")
hot = pd.read_csv(P + "hotellerie_monat.csv")
her = pd.read_csv(P + "herkunft_monat.csv")
wet = pd.read_csv(P + "wetter_monat.csv")
panel = pd.read_csv(P + "panel_winter.csv")
zeit = panel[["jahr", "monat", "winter", "winter_label", "covid", "ostern_im_monat"]].drop_duplicates()

print("zeit:", zeit.shape, "| doppelte Schlüssel:", zeit.duplicated(["jahr", "monat"]).sum())
for n, d in [("hotellerie", hot), ("herkunft", her), ("wetter", wet)]:
    print(n, "| Zeilen ohne Gemeinde:", (~d["bfs_nr"].isin(gem["bfs_nr"])).sum(),
          "| doppelte Schlüssel:", d.duplicated(["bfs_nr", "jahr", "monat"]).sum())

zeit: (52, 6) | doppelte Schlüssel: 0
hotellerie | Zeilen ohne Gemeinde: 0 | doppelte Schlüssel: 0
herkunft | Zeilen ohne Gemeinde: 0 | doppelte Schlüssel: 0
wetter | Zeilen ohne Gemeinde: 0 | doppelte Schlüssel: 0


In [5]:
tabellen = {"gemeinde": gem, "zeit": zeit, "hotellerie": hot, "herkunft": her, "wetter": wet}

def laden(eng):
    with eng.begin() as conn:
        for stmt in schema.split(";"):            # Schema neu anlegen, damit die Zelle wiederholbar ist
            if stmt.strip():
                conn.execute(text(stmt))
        for name, d in tabellen.items():          # Reihenfolge wegen Fremdschlüsseln
            d.to_sql(name, conn, if_exists="append", index=False, chunksize=5000)

laden(engine_sqlite)
print("SQLite fertig")
laden(engine_pg)
print("PostgreSQL fertig")

SQLite fertig
PostgreSQL fertig


In [6]:
zaehlen = text("""
SELECT 'gemeinde' AS tabelle, COUNT(*) AS zeilen FROM gemeinde
UNION ALL SELECT 'zeit', COUNT(*) FROM zeit
UNION ALL SELECT 'hotellerie', COUNT(*) FROM hotellerie
UNION ALL SELECT 'herkunft', COUNT(*) FROM herkunft
UNION ALL SELECT 'wetter', COUNT(*) FROM wetter
UNION ALL SELECT 'v_panel_winter', COUNT(*) FROM v_panel_winter
""")

for name, eng in [("SQLite", engine_sqlite), ("PostgreSQL", engine_pg)]:
    with eng.connect() as conn:
        z = pd.read_sql(zaehlen, conn)
        s = conn.execute(text("SELECT SUM(zimmerauslastung) FROM v_panel_winter")).scalar()
    print(name)
    print(z.to_string(index=False))
    print("Summe Zimmerauslastung (View):", round(s, 4), "| (CSV):", round(panel["zimmerauslastung"].sum(), 4), "\n")

SQLite
       tabelle  zeilen
      gemeinde     186
          zeit      52
    hotellerie   29459
      herkunft   29378
        wetter    7200
v_panel_winter    2255
Summe Zimmerauslastung (View): 120703.0 | (CSV): 120703.0 

PostgreSQL
       tabelle  zeilen
      gemeinde     186
          zeit      52
        wetter    7200
      herkunft   29378
    hotellerie   29459
v_panel_winter    2255
Summe Zimmerauslastung (View): 120703.0 | (CSV): 120703.0 



In [7]:
abfragen = {}

# 1) Join-Verlust: wo fehlen Herkunftsdaten (LEFT JOIN auf herkunft)?
abfragen["join_verlust"] = """
SELECT CASE WHEN g.ausgewaehlt THEN 'ausgewählt' ELSE 'nicht ausgewählt' END AS gruppe,
       COUNT(*) AS hotellerie_zeilen,
       SUM(CASE WHEN k.bfs_nr IS NULL THEN 1 ELSE 0 END) AS ohne_herkunft,
       SUM(CASE WHEN h.zimmerauslastung IS NULL THEN 1 ELSE 0 END) AS ohne_zimmerauslastung
FROM hotellerie h
JOIN gemeinde g ON g.bfs_nr = h.bfs_nr
LEFT JOIN herkunft k ON k.bfs_nr = h.bfs_nr AND k.jahr = h.jahr AND k.monat = h.monat
GROUP BY g.ausgewaehlt
ORDER BY gruppe
"""

# 2) Lücken: Gemeinde x Wintermonat, zu dem es keine BFS-Zeile gibt (CROSS JOIN + LEFT JOIN)
abfragen["luecken"] = """
SELECT g.bfs_nr, g.name, z.winter, z.winter_label, COUNT(*) AS fehlende_monate
FROM gemeinde g
CROSS JOIN zeit z
LEFT JOIN hotellerie h ON h.bfs_nr = g.bfs_nr AND h.jahr = z.jahr AND h.monat = z.monat
WHERE g.ausgewaehlt AND h.bfs_nr IS NULL
GROUP BY g.bfs_nr, g.name, z.winter, z.winter_label
ORDER BY g.bfs_nr, z.winter
"""

# 3) Aggregation: Auslastung, Schnee, Temperatur je Höhenklasse und Winter
abfragen["hoehenklasse_winter"] = """
SELECT hoehenklasse, winter, winter_label,
       COUNT(*) AS n,
       AVG(zimmerauslastung) AS auslastung_mittel,
       AVG(schneefall_cm) AS schnee_mittel,
       AVG(temp_mittel) AS temp_mittel
FROM v_panel_winter
GROUP BY hoehenklasse, winter, winter_label
ORDER BY hoehenklasse, winter
"""

# 4) Aggregation je Gemeinde
abfragen["gemeinde_profil"] = """
SELECT bfs_nr, name, hoehe_m, hoehenklasse,
       COUNT(*) AS n,
       AVG(zimmerauslastung) AS auslastung_mittel,
       AVG(schneefall_cm) AS schnee_mittel
FROM v_panel_winter
GROUP BY bfs_nr, name, hoehe_m, hoehenklasse
ORDER BY bfs_nr
"""

def abfrage(sql, eng):
    with eng.connect() as conn:
        return pd.read_sql(text(sql), conn)

ergebnisse = {}
for name, sql in abfragen.items():
    a = abfrage(sql, engine_sqlite)
    b = abfrage(sql, engine_pg)
    pd.testing.assert_frame_equal(a, b, check_dtype=False, rtol=1e-6)   # beide DBs müssen dasselbe liefern
    ergebnisse[name] = a
    print(f"{name}: {len(a)} Zeilen, SQLite = PostgreSQL")

join_verlust: 2 Zeilen, SQLite = PostgreSQL
luecken: 25 Zeilen, SQLite = PostgreSQL
hoehenklasse_winter: 39 Zeilen, SQLite = PostgreSQL
gemeinde_profil: 45 Zeilen, SQLite = PostgreSQL


In [8]:
r = ergebnisse
display(r["join_verlust"])
print("Fehlende Gemeinde-Wintermonate:", r["luecken"]["fehlende_monate"].sum())
display(r["luecken"].groupby("name")["fehlende_monate"].sum())
display(r["hoehenklasse_winter"].round(2).pivot(index="winter_label", columns="hoehenklasse", values="auslastung_mittel"))

,gruppe,hotellerie_zeilen,ohne_herkunft,ohne_zimmerauslastung
0,ausgewählt,7080,26,26
1,nicht ausgewählt,22379,55,55


Fehlende Gemeinde-Wintermonate: 85


name
Crans-Montana    13
Goms             13
Rheinwald        21
Surses            9
Val de Bagnes    29
Name: fehlende_monate, dtype: int64

hoehenklasse,hoch,mittel,tief
winter_label,,,
2013/14,63.43,55.97,46.98
2014/15,63.23,52.94,46.42
2015/16,59.59,51.12,45.67
2016/17,60.21,49.80,43.85
2017/18,63.65,53.65,46.00
2018/19,63.29,53.12,46.71
2019/20,59.12,50.17,45.17
2020/21,48.22,46.57,38.29
2021/22,63.52,57.12,49.85


- Unterdrückte Werte (`...`): 81 Hotellerie-Zeilen ohne Zimmerauslastung und ohne Herkunft (26 bei den gewählten Gemeinden, 55 bei den übrigen). Die 26 liegen ausserhalb des Analysefensters (Dez–Mär, Winter 2013/14–2025/26), im Panel fehlen daher keine Werte.
- Lücken: 85 von 2'340 möglichen Gemeinde-Wintermonaten fehlen in den BFS-Daten ganz: Val de Bagnes 29, Rheinwald 21, Crans-Montana 13, Goms 13, Surses 9, ausschliesslich in den Wintern 2013/14 bis 2020/21. Ab 2021/22 ist das Panel vollständig. Das Panel bleibt unbalanciert, es wird nichts imputiert. Ursache nicht abschliessend geklärt.
- 3 Zeilen (Davos, 2014) weichen in den Logiernächten zwischen Tabelle 201 und 101 ab (Differenz: ...).